# Build Earnings Straddle Dataset — Corrected Minimal Version

This version keeps your original workflow but adds the important corrections only:

1. Cleans option quotes inside this notebook.
2. Chooses ATM only from strikes with both call and put.
3. Fixes CRSP negative-price realized move calculation.
4. Creates both simple after-expiry implied move and isolated implied move.
5. Adds trading edge = realized move - implied move.
6. Saves loose and strict clean datasets.

In [1]:

import duckdb
import os
from pathlib import Path

con = duckdb.connect()

# ============================================================
# PATHS
# ============================================================
PROCESSED_DIR = r"data\processed"

earnings_prices_clean_file = os.path.join(PROCESSED_DIR, "earnings_prices_clean.parquet")
name_file = os.path.join(PROCESSED_DIR, "security_name_history_small.parquet")
crsp_file = os.path.join(PROCESSED_DIR, "crsp_daily_stock_small_v2.parquet")
sec_price_file = os.path.join(PROCESSED_DIR, "security_prices_small.parquet")
opt_short_file = os.path.join(PROCESSED_DIR, "option_prices_earnings_short_v2.parquet")

# Existing output name, kept for compatibility with your next cells
output_file = os.path.join(PROCESSED_DIR, "earnings_straddles_choiceA.parquet")

# New optional output: larger after-expiry-only dataset
output_file_simple = os.path.join(PROCESSED_DIR, "earnings_straddles_simple_after.parquet")

# Delete output files first so COPY does not fail if file already exists
for f in [output_file, output_file_simple]:
    if os.path.exists(f):
        os.remove(f)

# ============================================================
# Helper
# ============================================================
def show(title, query, n=20):
    print("\n" + "=" * 100)
    print(title)
    print("=" * 100)
    df = con.execute(query).df()
    if len(df) > n:
        print(df.head(n).to_string(index=False))
        print(f"\n[showing first {n} rows out of {len(df)}]")
    else:
        print(df.to_string(index=False))

# ============================================================
# STEP 1 — Load earnings sample
# ============================================================
con.execute(f"""
CREATE OR REPLACE TEMP TABLE earnings_prices_clean AS
SELECT *
FROM read_parquet('{earnings_prices_clean_file}')
""")

# ============================================================
# STEP 2 — Map earnings events to OptionMetrics secid
# ============================================================
con.execute(f"""
CREATE OR REPLACE TEMP TABLE crsp_id_history AS
SELECT DISTINCT
    permno,
    date,
    NCUSIP AS ncusip,
    TICKER AS crsp_ticker
FROM read_parquet('{crsp_file}')
WHERE NCUSIP IS NOT NULL
""")

con.execute("""
CREATE OR REPLACE TEMP TABLE event_crsp_id AS
SELECT
    e.*,
    c.ncusip,
    c.crsp_ticker
FROM earnings_prices_clean e
LEFT JOIN crsp_id_history c
  ON e.permno = c.permno
 AND e.prev_date = c.date
""")

con.execute(f"""
CREATE OR REPLACE TEMP TABLE om_name_history AS
SELECT
    secid,
    ticker AS om_ticker,
    cusip AS om_cusip,
    effect_date
FROM read_parquet('{name_file}')
""")

con.execute("""
CREATE OR REPLACE TEMP TABLE event_secid AS
SELECT *
FROM (
    SELECT
        e.*,
        o.secid,
        o.om_ticker,
        o.om_cusip,
        o.effect_date,
        ROW_NUMBER() OVER (
            PARTITION BY e.gvkey, e.rdq
            ORDER BY o.effect_date DESC, o.secid
        ) AS rn
    FROM event_crsp_id e
    LEFT JOIN om_name_history o
      ON e.ncusip = o.om_cusip
     AND o.effect_date <= e.prev_date
) t
WHERE rn = 1
  AND secid IS NOT NULL
""")

# ============================================================
# STEP 3 — Attach OptionMetrics underlying price on prev_date
# ============================================================
con.execute(f"""
CREATE OR REPLACE TEMP TABLE om_sec_prices AS
SELECT
    secid,
    date,
    close AS om_close,
    open AS om_open,
    ticker AS om_price_ticker,
    cusip AS om_price_cusip
FROM read_parquet('{sec_price_file}')
""")

con.execute("""
CREATE OR REPLACE TEMP TABLE event_ready AS
SELECT
    e.*,
    s.om_close,
    s.om_open,
    s.om_price_ticker,
    s.om_price_cusip
FROM event_secid e
LEFT JOIN om_sec_prices s
  ON e.secid = s.secid
 AND e.prev_date = s.date
WHERE s.om_close IS NOT NULL
  AND TRY_CAST(s.om_close AS DOUBLE) IS NOT NULL
  AND TRY_CAST(s.om_close AS DOUBLE) > 0
""")

# ============================================================
# STEP 4 — Clean option quotes inside this notebook
# This avoids editing older notebooks.
# ============================================================
con.execute(f"""
CREATE OR REPLACE TEMP TABLE opt_valid AS
SELECT
    secid,
    date,
    exdate,
    cp_flag,
    strike_price,
    best_bid,
    best_offer,
    open_interest,
    volume,
    (TRY_CAST(best_bid AS DOUBLE) + TRY_CAST(best_offer AS DOUBLE)) / 2.0 AS mid
FROM read_parquet('{opt_short_file}')
WHERE best_bid IS NOT NULL
  AND best_offer IS NOT NULL
  AND strike_price IS NOT NULL
  AND cp_flag IN ('C', 'P')
  AND TRY_CAST(best_bid AS DOUBLE) >= 0
  AND TRY_CAST(best_offer AS DOUBLE) > 0
  AND TRY_CAST(best_offer AS DOUBLE) >= TRY_CAST(best_bid AS DOUBLE)
  AND (TRY_CAST(best_bid AS DOUBLE) + TRY_CAST(best_offer AS DOUBLE)) / 2.0 > 0
""")

# ============================================================
# STEP 5 — Build option chain dates from cleaned short-DTE options
# ============================================================
con.execute("""
CREATE OR REPLACE TEMP TABLE option_chain_dates AS
SELECT DISTINCT
    secid,
    date,
    exdate
FROM opt_valid
""")

# ============================================================
# STEP 6 — Find before/after expiries around rdq
# Isolated version needs both before and after.
# Simple version only needs after.
# ============================================================
con.execute("""
CREATE OR REPLACE TEMP TABLE event_expiry_before AS
SELECT *
FROM (
    SELECT
        e.gvkey,
        e.rdq,
        e.secid,
        e.prev_date,
        c.exdate AS before_exdate,
        ROW_NUMBER() OVER (
            PARTITION BY e.gvkey, e.rdq
            ORDER BY c.exdate DESC
        ) AS rn
    FROM event_ready e
    JOIN option_chain_dates c
      ON e.secid = c.secid
     AND e.prev_date = c.date
    WHERE c.exdate <= e.rdq
) t
WHERE rn = 1
""")

con.execute("""
CREATE OR REPLACE TEMP TABLE event_expiry_after AS
SELECT *
FROM (
    SELECT
        e.gvkey,
        e.rdq,
        e.secid,
        e.prev_date,
        c.exdate AS after_exdate,
        ROW_NUMBER() OVER (
            PARTITION BY e.gvkey, e.rdq
            ORDER BY c.exdate ASC
        ) AS rn
    FROM event_ready e
    JOIN option_chain_dates c
      ON e.secid = c.secid
     AND e.prev_date = c.date
    WHERE c.exdate > e.rdq
) t
WHERE rn = 1
""")

# Simple version: only needs after expiry
con.execute("""
CREATE OR REPLACE TEMP TABLE event_after_only AS
SELECT
    e.*,
    a.after_exdate
FROM event_ready e
JOIN event_expiry_after a
  ON e.gvkey = a.gvkey 
 AND e.rdq = a.rdq
""")

# Isolated version: needs before and after expiry
con.execute("""
CREATE OR REPLACE TEMP TABLE event_expiries AS
SELECT
    e.*,
    b.before_exdate,
    a.after_exdate
FROM event_ready e
JOIN event_expiry_before b
  ON e.gvkey = b.gvkey 
 AND e.rdq = b.rdq
JOIN event_expiry_after a
  ON e.gvkey = a.gvkey 
 AND e.rdq = a.rdq
""")

# ============================================================
# STEP 7 — Choose ATM strike from AFTER expiry,
# but only among strikes with BOTH call and put.
# ============================================================

# For simple after-only dataset
con.execute("""
CREATE OR REPLACE TEMP TABLE after_pairable_strikes_simple AS
SELECT
    e.gvkey,
    e.tic,
    e.permno,
    e.secid,
    e.rdq,
    e.prev_date,
    e.prev_prc,
    e.same_date,
    e.same_prc,
    e.next_date,
    e.next_prc,
    e.prev_openprc,
    e.same_openprc,
    e.next_openprc,
    e.om_close,
    e.after_exdate,
    o.strike_price,
    o.strike_price / 1000.0 AS strike,
    ABS(o.strike_price / 1000.0 - e.om_close) AS strike_distance,
    MAX(CASE WHEN o.cp_flag = 'C' THEN 1 ELSE 0 END) AS has_call,
    MAX(CASE WHEN o.cp_flag = 'P' THEN 1 ELSE 0 END) AS has_put
FROM event_after_only e
JOIN opt_valid o
  ON e.secid = o.secid
 AND e.prev_date = o.date
 AND e.after_exdate = o.exdate
GROUP BY
    e.gvkey, e.tic, e.permno, e.secid, e.rdq, e.prev_date,
    e.prev_prc, e.same_date, e.same_prc, e.next_date, e.next_prc,
    e.prev_openprc, e.same_openprc, e.next_openprc,
    e.om_close, e.after_exdate, o.strike_price
HAVING has_call = 1 AND has_put = 1
""")

con.execute("""
CREATE OR REPLACE TEMP TABLE atm_choice_simple AS
SELECT *
FROM (
    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY gvkey, rdq
            ORDER BY strike_distance ASC, strike ASC
        ) AS rn
    FROM after_pairable_strikes_simple
) t
WHERE rn = 1
""")

# For isolated dataset
con.execute("""
CREATE OR REPLACE TEMP TABLE after_pairable_strikes_isolated AS
SELECT
    e.gvkey,
    e.tic,
    e.permno,
    e.secid,
    e.rdq,
    e.prev_date,
    e.prev_prc,
    e.same_date,
    e.same_prc,
    e.next_date,
    e.next_prc,
    e.prev_openprc,
    e.same_openprc,
    e.next_openprc,
    e.om_close,
    e.before_exdate,
    e.after_exdate,
    o.strike_price,
    o.strike_price / 1000.0 AS strike,
    ABS(o.strike_price / 1000.0 - e.om_close) AS strike_distance,
    MAX(CASE WHEN o.cp_flag = 'C' THEN 1 ELSE 0 END) AS has_call,
    MAX(CASE WHEN o.cp_flag = 'P' THEN 1 ELSE 0 END) AS has_put
FROM event_expiries e
JOIN opt_valid o
  ON e.secid = o.secid
 AND e.prev_date = o.date
 AND e.after_exdate = o.exdate
GROUP BY
    e.gvkey, e.tic, e.permno, e.secid, e.rdq, e.prev_date,
    e.prev_prc, e.same_date, e.same_prc, e.next_date, e.next_prc,
    e.prev_openprc, e.same_openprc, e.next_openprc,
    e.om_close, e.before_exdate, e.after_exdate, o.strike_price
HAVING has_call = 1 AND has_put = 1
""")

con.execute("""
CREATE OR REPLACE TEMP TABLE atm_choice AS
SELECT *
FROM (
    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY gvkey, rdq
            ORDER BY strike_distance ASC, strike ASC
        ) AS rn
    FROM after_pairable_strikes_isolated
) t
WHERE rn = 1
""")

# ============================================================
# STEP 8 — Pull after-expiry call/put legs
# ============================================================

con.execute("""
CREATE OR REPLACE TEMP TABLE after_legs_simple AS
SELECT
    a.gvkey,
    a.rdq,
    MAX(CASE WHEN o.cp_flag = 'C' THEN o.mid END) AS after_call_mid,
    MAX(CASE WHEN o.cp_flag = 'P' THEN o.mid END) AS after_put_mid,
    MAX(CASE WHEN o.cp_flag = 'C' THEN o.best_bid END) AS after_call_bid,
    MAX(CASE WHEN o.cp_flag = 'C' THEN o.best_offer END) AS after_call_ask,
    MAX(CASE WHEN o.cp_flag = 'P' THEN o.best_bid END) AS after_put_bid,
    MAX(CASE WHEN o.cp_flag = 'P' THEN o.best_offer END) AS after_put_ask,
    MAX(CASE WHEN o.cp_flag = 'C' THEN o.open_interest END) AS after_call_oi,
    MAX(CASE WHEN o.cp_flag = 'P' THEN o.open_interest END) AS after_put_oi,
    MAX(CASE WHEN o.cp_flag = 'C' THEN o.volume END) AS after_call_vol,
    MAX(CASE WHEN o.cp_flag = 'P' THEN o.volume END) AS after_put_vol
FROM atm_choice_simple a
LEFT JOIN opt_valid o
  ON a.secid = o.secid
 AND a.prev_date = o.date
 AND a.after_exdate = o.exdate
 AND a.strike_price = o.strike_price
GROUP BY a.gvkey, a.rdq
""")

con.execute("""
CREATE OR REPLACE TEMP TABLE after_legs AS
SELECT
    a.gvkey,
    a.rdq,
    MAX(CASE WHEN o.cp_flag = 'C' THEN o.mid END) AS after_call_mid,
    MAX(CASE WHEN o.cp_flag = 'P' THEN o.mid END) AS after_put_mid,
    MAX(CASE WHEN o.cp_flag = 'C' THEN o.best_bid END) AS after_call_bid,
    MAX(CASE WHEN o.cp_flag = 'C' THEN o.best_offer END) AS after_call_ask,
    MAX(CASE WHEN o.cp_flag = 'P' THEN o.best_bid END) AS after_put_bid,
    MAX(CASE WHEN o.cp_flag = 'P' THEN o.best_offer END) AS after_put_ask,
    MAX(CASE WHEN o.cp_flag = 'C' THEN o.open_interest END) AS after_call_oi,
    MAX(CASE WHEN o.cp_flag = 'P' THEN o.open_interest END) AS after_put_oi,
    MAX(CASE WHEN o.cp_flag = 'C' THEN o.volume END) AS after_call_vol,
    MAX(CASE WHEN o.cp_flag = 'P' THEN o.volume END) AS after_put_vol
FROM atm_choice a
LEFT JOIN opt_valid o
  ON a.secid = o.secid
 AND a.prev_date = o.date
 AND a.after_exdate = o.exdate
 AND a.strike_price = o.strike_price
GROUP BY a.gvkey, a.rdq
""")

# ============================================================
# STEP 9 — Pull before-expiry call/put legs at same strike
# Only needed for isolated version.
# ============================================================
con.execute("""
CREATE OR REPLACE TEMP TABLE before_legs AS
SELECT
    a.gvkey,
    a.rdq,
    MAX(CASE WHEN o.cp_flag = 'C' THEN o.mid END) AS before_call_mid,
    MAX(CASE WHEN o.cp_flag = 'P' THEN o.mid END) AS before_put_mid,
    MAX(CASE WHEN o.cp_flag = 'C' THEN o.best_bid END) AS before_call_bid,
    MAX(CASE WHEN o.cp_flag = 'C' THEN o.best_offer END) AS before_call_ask,
    MAX(CASE WHEN o.cp_flag = 'P' THEN o.best_bid END) AS before_put_bid,
    MAX(CASE WHEN o.cp_flag = 'P' THEN o.best_offer END) AS before_put_ask,
    MAX(CASE WHEN o.cp_flag = 'C' THEN o.open_interest END) AS before_call_oi,
    MAX(CASE WHEN o.cp_flag = 'P' THEN o.open_interest END) AS before_put_oi,
    MAX(CASE WHEN o.cp_flag = 'C' THEN o.volume END) AS before_call_vol,
    MAX(CASE WHEN o.cp_flag = 'P' THEN o.volume END) AS before_put_vol
FROM atm_choice a
LEFT JOIN opt_valid o
  ON a.secid = o.secid
 AND a.prev_date = o.date
 AND a.before_exdate = o.exdate
 AND a.strike_price = o.strike_price
GROUP BY a.gvkey, a.rdq
""")

# ============================================================
# STEP 10 — Build larger simple after-expiry straddle dataset
# ============================================================
con.execute("""
CREATE OR REPLACE TEMP TABLE earnings_straddles_simple_after AS
SELECT
    a.gvkey,
    a.tic,
    a.permno,
    a.secid,
    a.rdq,
    a.prev_date,
    NULL::DATE AS before_exdate,
    a.after_exdate,
    a.om_close,
    a.strike,
    a.strike_price,

    NULL::DOUBLE AS before_call_mid,
    NULL::DOUBLE AS before_put_mid,
    NULL::DOUBLE AS before_straddle_mid,

    af.after_call_mid,
    af.after_put_mid,
    (af.after_call_mid + af.after_put_mid) AS after_straddle_mid,

    NULL::DOUBLE AS before_call_bid,
    NULL::DOUBLE AS before_call_ask,
    NULL::DOUBLE AS before_put_bid,
    NULL::DOUBLE AS before_put_ask,
    af.after_call_bid,
    af.after_call_ask,
    af.after_put_bid,
    af.after_put_ask,

    NULL::DOUBLE AS before_call_oi,
    NULL::DOUBLE AS before_put_oi,
    af.after_call_oi,
    af.after_put_oi,

    NULL::DOUBLE AS before_call_vol,
    NULL::DOUBLE AS before_put_vol,
    af.after_call_vol,
    af.after_put_vol,

    a.prev_prc,
    a.same_date,
    a.same_prc,
    a.next_date,
    a.next_prc,
    a.prev_openprc,
    a.same_openprc,
    a.next_openprc,

    CASE
        WHEN TRY_CAST(a.om_close AS DOUBLE) > 0
         AND af.after_call_mid IS NOT NULL
         AND af.after_put_mid IS NOT NULL
        THEN (af.after_call_mid + af.after_put_mid) / TRY_CAST(a.om_close AS DOUBLE)
        ELSE NULL
    END AS implied_move_simple,

    NULL::DOUBLE AS implied_move_isolated,

    CASE
        WHEN TRY_CAST(a.prev_prc AS DOUBLE) IS NOT NULL
         AND ABS(TRY_CAST(a.prev_prc AS DOUBLE)) <> 0
         AND TRY_CAST(a.next_openprc AS DOUBLE) IS NOT NULL
        THEN ABS(
            ABS(TRY_CAST(a.next_openprc AS DOUBLE)) -
            ABS(TRY_CAST(a.prev_prc AS DOUBLE))
        ) / ABS(TRY_CAST(a.prev_prc AS DOUBLE))
        ELSE NULL
    END AS realized_move_open_to_prevclose,

    CASE
        WHEN TRY_CAST(a.prev_prc AS DOUBLE) IS NOT NULL
         AND ABS(TRY_CAST(a.prev_prc AS DOUBLE)) <> 0
         AND TRY_CAST(a.next_prc AS DOUBLE) IS NOT NULL
        THEN ABS(
            ABS(TRY_CAST(a.next_prc AS DOUBLE)) -
            ABS(TRY_CAST(a.prev_prc AS DOUBLE))
        ) / ABS(TRY_CAST(a.prev_prc AS DOUBLE))
        ELSE NULL
    END AS realized_move_close_to_prevclose

FROM atm_choice_simple a
LEFT JOIN after_legs_simple af
  ON a.gvkey = af.gvkey 
 AND a.rdq = af.rdq
""")

# ============================================================
# STEP 11 — Build isolated Choice A dataset
# ============================================================
con.execute("""
CREATE OR REPLACE TEMP TABLE earnings_straddles_choiceA AS
SELECT
    a.gvkey,
    a.tic,
    a.permno,
    a.secid,
    a.rdq,
    a.prev_date,
    a.before_exdate,
    a.after_exdate,
    a.om_close,
    a.strike,
    a.strike_price,

    b.before_call_mid,
    b.before_put_mid,
    (b.before_call_mid + b.before_put_mid) AS before_straddle_mid,

    af.after_call_mid,
    af.after_put_mid,
    (af.after_call_mid + af.after_put_mid) AS after_straddle_mid,

    b.before_call_bid,
    b.before_call_ask,
    b.before_put_bid,
    b.before_put_ask,
    af.after_call_bid,
    af.after_call_ask,
    af.after_put_bid,
    af.after_put_ask,

    b.before_call_oi,
    b.before_put_oi,
    af.after_call_oi,
    af.after_put_oi,

    b.before_call_vol,
    b.before_put_vol,
    af.after_call_vol,
    af.after_put_vol,

    a.prev_prc,
    a.same_date,
    a.same_prc,
    a.next_date,
    a.next_prc,
    a.prev_openprc,
    a.same_openprc,
    a.next_openprc,

    CASE
        WHEN TRY_CAST(a.om_close AS DOUBLE) > 0
         AND af.after_call_mid IS NOT NULL
         AND af.after_put_mid IS NOT NULL
        THEN (af.after_call_mid + af.after_put_mid) / TRY_CAST(a.om_close AS DOUBLE)
        ELSE NULL
    END AS implied_move_simple,

    CASE
        WHEN TRY_CAST(a.om_close AS DOUBLE) > 0
         AND af.after_call_mid IS NOT NULL
         AND af.after_put_mid IS NOT NULL
         AND b.before_call_mid IS NOT NULL
         AND b.before_put_mid IS NOT NULL
        THEN (
            (af.after_call_mid + af.after_put_mid)
            - (b.before_call_mid + b.before_put_mid)
        ) / TRY_CAST(a.om_close AS DOUBLE)
        ELSE NULL
    END AS implied_move_isolated,

    CASE
        WHEN TRY_CAST(a.prev_prc AS DOUBLE) IS NOT NULL
         AND ABS(TRY_CAST(a.prev_prc AS DOUBLE)) <> 0
         AND TRY_CAST(a.next_openprc AS DOUBLE) IS NOT NULL
        THEN ABS(
            ABS(TRY_CAST(a.next_openprc AS DOUBLE)) -
            ABS(TRY_CAST(a.prev_prc AS DOUBLE))
        ) / ABS(TRY_CAST(a.prev_prc AS DOUBLE))
        ELSE NULL
    END AS realized_move_open_to_prevclose,

    CASE
        WHEN TRY_CAST(a.prev_prc AS DOUBLE) IS NOT NULL
         AND ABS(TRY_CAST(a.prev_prc AS DOUBLE)) <> 0
         AND TRY_CAST(a.next_prc AS DOUBLE) IS NOT NULL
        THEN ABS(
            ABS(TRY_CAST(a.next_prc AS DOUBLE)) -
            ABS(TRY_CAST(a.prev_prc AS DOUBLE))
        ) / ABS(TRY_CAST(a.prev_prc AS DOUBLE))
        ELSE NULL
    END AS realized_move_close_to_prevclose

FROM atm_choice a
LEFT JOIN before_legs b
  ON a.gvkey = b.gvkey 
 AND a.rdq = b.rdq
LEFT JOIN after_legs af
  ON a.gvkey = af.gvkey 
 AND a.rdq = af.rdq
""")

# ============================================================
# STEP 12 — Save both datasets
# ============================================================
con.execute(f"""
COPY earnings_straddles_simple_after 
TO '{output_file_simple}' 
(FORMAT PARQUET)
""")

con.execute(f"""
COPY earnings_straddles_choiceA 
TO '{output_file}' 
(FORMAT PARQUET)
""")

print(f"\nSaved simple after-expiry dataset: {output_file_simple}")
print(f"Saved isolated Choice A dataset: {output_file}")

# ============================================================
# CHECKS
# ============================================================
show(
    "1. Events with after expiry only",
    "SELECT COUNT(*) AS n_rows FROM event_after_only"
)

show(
    "2. Events with both before and after expiries",
    "SELECT COUNT(*) AS n_rows FROM event_expiries"
)

show(
    "3. ATM choice row counts",
    """
    SELECT
        (SELECT COUNT(*) FROM atm_choice_simple) AS simple_after_atm_rows,
        (SELECT COUNT(*) FROM atm_choice) AS isolated_atm_rows
    """
)

show(
    "4. Straddle completeness: simple after-expiry version",
    """
    SELECT
        COUNT(*) AS total_rows,
        SUM(CASE WHEN after_straddle_mid IS NOT NULL THEN 1 ELSE 0 END) AS rows_with_after_straddle,
        SUM(CASE WHEN implied_move_simple IS NOT NULL THEN 1 ELSE 0 END) AS rows_with_implied_move_simple,
        SUM(CASE WHEN realized_move_open_to_prevclose IS NOT NULL THEN 1 ELSE 0 END) AS rows_with_realized_open
    FROM earnings_straddles_simple_after
    """
)

show(
    "5. Straddle completeness: isolated Choice A version",
    """
    SELECT
        COUNT(*) AS total_rows,
        SUM(CASE WHEN before_straddle_mid IS NOT NULL THEN 1 ELSE 0 END) AS rows_with_before_straddle,
        SUM(CASE WHEN after_straddle_mid IS NOT NULL THEN 1 ELSE 0 END) AS rows_with_after_straddle,
        SUM(CASE WHEN implied_move_isolated IS NOT NULL THEN 1 ELSE 0 END) AS rows_with_implied_move_isolated,
        SUM(CASE WHEN realized_move_open_to_prevclose IS NOT NULL THEN 1 ELSE 0 END) AS rows_with_realized_open
    FROM earnings_straddles_choiceA
    """
)

show(
    "6. Check for negative CRSP prices in final isolated rows",
    """
    SELECT
        COUNT(*) AS total_rows,
        SUM(CASE WHEN TRY_CAST(prev_prc AS DOUBLE) < 0 THEN 1 ELSE 0 END) AS negative_prev_prc,
        SUM(CASE WHEN TRY_CAST(next_openprc AS DOUBLE) < 0 THEN 1 ELSE 0 END) AS negative_next_openprc,
        SUM(CASE WHEN TRY_CAST(next_prc AS DOUBLE) < 0 THEN 1 ELSE 0 END) AS negative_next_prc
    FROM earnings_straddles_choiceA
    """
)

show(
    "7. Sample isolated Choice A rows",
    """
    SELECT
        tic,
        rdq,
        prev_date,
        before_exdate,
        after_exdate,
        om_close,
        strike,
        before_straddle_mid,
        after_straddle_mid,
        implied_move_simple,
        implied_move_isolated,
        realized_move_open_to_prevclose,
        realized_move_close_to_prevclose
    FROM earnings_straddles_choiceA
    WHERE implied_move_isolated IS NOT NULL
    ORDER BY rdq, tic
    LIMIT 30
    """
)



Saved simple after-expiry dataset: data\processed\earnings_straddles_simple_after.parquet
Saved isolated Choice A dataset: data\processed\earnings_straddles_choiceA.parquet

1. Events with after expiry only
 n_rows
   9804

2. Events with both before and after expiries
 n_rows
    967

3. ATM choice row counts
 simple_after_atm_rows  isolated_atm_rows
                  9804                967

4. Straddle completeness: simple after-expiry version
 total_rows  rows_with_after_straddle  rows_with_implied_move_simple  rows_with_realized_open
       9804                    9804.0                         9804.0                   9804.0

5. Straddle completeness: isolated Choice A version
 total_rows  rows_with_before_straddle  rows_with_after_straddle  rows_with_implied_move_isolated  rows_with_realized_open
        967                      955.0                     967.0                            955.0                    967.0

6. Check for negative CRSP prices in final isolated rows
 to

In [2]:

import duckdb
import os

con = duckdb.connect()

PROCESSED_DIR = r"data\processed"

# You can switch this to earnings_straddles_simple_after.parquet if you want the larger simple version.
# For now, this keeps your original isolated Choice A file as the base.
choiceA_file = os.path.join(PROCESSED_DIR, "earnings_straddles_choiceA.parquet")
final_base_file = os.path.join(PROCESSED_DIR, "earnings_model_base.parquet")

if os.path.exists(final_base_file):
    os.remove(final_base_file)

con.execute(f"""
CREATE OR REPLACE TEMP TABLE choiceA AS
SELECT *
FROM read_parquet('{choiceA_file}')
""")

# ------------------------------------------------------------------
# Final modeling base:
# Keep both implied_move_simple and implied_move_isolated.
# Add trading edge = realized move - implied move.
# ------------------------------------------------------------------
con.execute("""
CREATE OR REPLACE TEMP TABLE earnings_model_base AS
SELECT
    gvkey,
    tic,
    permno,
    secid,
    rdq,
    prev_date,
    before_exdate,
    after_exdate,
    om_close,
    strike,
    strike_price,

    before_call_mid,
    before_put_mid,
    before_straddle_mid,

    after_call_mid,
    after_put_mid,
    after_straddle_mid,

    before_call_bid,
    before_call_ask,
    before_put_bid,
    before_put_ask,
    after_call_bid,
    after_call_ask,
    after_put_bid,
    after_put_ask,

    before_call_oi,
    before_put_oi,
    after_call_oi,
    after_put_oi,

    before_call_vol,
    before_put_vol,
    after_call_vol,
    after_put_vol,

    implied_move_simple,
    implied_move_isolated,
    realized_move_open_to_prevclose,
    realized_move_close_to_prevclose,

    -- Old sign, kept for reference:
    implied_move_isolated - realized_move_open_to_prevclose AS mispricing_open,
    implied_move_isolated - realized_move_close_to_prevclose AS mispricing_close,

    -- Better target sign for trading:
    realized_move_open_to_prevclose - implied_move_isolated AS straddle_edge_open,
    realized_move_close_to_prevclose - implied_move_isolated AS straddle_edge_close,

    realized_move_open_to_prevclose - implied_move_simple AS simple_straddle_edge_open,
    realized_move_close_to_prevclose - implied_move_simple AS simple_straddle_edge_close,

    (before_call_ask - before_call_bid) AS before_call_spread,
    (before_put_ask - before_put_bid) AS before_put_spread,
    (after_call_ask - after_call_bid) AS after_call_spread,
    (after_put_ask - after_put_bid) AS after_put_spread,

    CASE
        WHEN before_call_mid IS NOT NULL AND before_call_mid <> 0
        THEN (before_call_ask - before_call_bid) / before_call_mid
        ELSE NULL
    END AS before_call_spread_pct,

    CASE
        WHEN before_put_mid IS NOT NULL AND before_put_mid <> 0
        THEN (before_put_ask - before_put_bid) / before_put_mid
        ELSE NULL
    END AS before_put_spread_pct,

    CASE
        WHEN after_call_mid IS NOT NULL AND after_call_mid <> 0
        THEN (after_call_ask - after_call_bid) / after_call_mid
        ELSE NULL
    END AS after_call_spread_pct,

    CASE
        WHEN after_put_mid IS NOT NULL AND after_put_mid <> 0
        THEN (after_put_ask - after_put_bid) / after_put_mid
        ELSE NULL
    END AS after_put_spread_pct

FROM choiceA
WHERE implied_move_isolated IS NOT NULL
  AND realized_move_open_to_prevclose IS NOT NULL
""")

con.execute(f"""
COPY earnings_model_base 
TO '{final_base_file}' 
(FORMAT PARQUET)
""")

print(f"Saved: {final_base_file}")

def show(title, query, n=20):
    print("\n" + "=" * 100)
    print(title)
    print("=" * 100)
    df = con.execute(query).df()
    if len(df) > n:
        print(df.head(n).to_string(index=False))
        print(f"\n[showing first {n} rows out of {len(df)}]")
    else:
        print(df.to_string(index=False))

show(
    "1. Final modeling base row count",
    "SELECT COUNT(*) AS n_rows FROM earnings_model_base"
)

show(
    "2. Distribution of implied move / realized move / edge",
    """
    SELECT
        AVG(implied_move_simple) AS avg_implied_move_simple,
        AVG(implied_move_isolated) AS avg_implied_move_isolated,
        AVG(realized_move_open_to_prevclose) AS avg_realized_move_open,
        AVG(straddle_edge_open) AS avg_straddle_edge_open,
        AVG(realized_move_close_to_prevclose) AS avg_realized_move_close,
        AVG(straddle_edge_close) AS avg_straddle_edge_close
    FROM earnings_model_base
    """
)

show(
    "3. Quantiles of straddle_edge_open",
    """
    SELECT
        MIN(straddle_edge_open) AS min_edge,
        APPROX_QUANTILE(straddle_edge_open, 0.05) AS p05,
        APPROX_QUANTILE(straddle_edge_open, 0.25) AS p25,
        APPROX_QUANTILE(straddle_edge_open, 0.50) AS median,
        APPROX_QUANTILE(straddle_edge_open, 0.75) AS p75,
        APPROX_QUANTILE(straddle_edge_open, 0.95) AS p95,
        MAX(straddle_edge_open) AS max_edge
    FROM earnings_model_base
    """
)

show(
    "4. Spread diagnostics",
    """
    SELECT
        AVG(before_call_spread_pct) AS avg_before_call_spread_pct,
        AVG(before_put_spread_pct)  AS avg_before_put_spread_pct,
        AVG(after_call_spread_pct)  AS avg_after_call_spread_pct,
        AVG(after_put_spread_pct)   AS avg_after_put_spread_pct
    FROM earnings_model_base
    """
)


Saved: data\processed\earnings_model_base.parquet

1. Final modeling base row count
 n_rows
    955

2. Distribution of implied move / realized move / edge
 avg_implied_move_simple  avg_implied_move_isolated  avg_realized_move_open  avg_straddle_edge_open  avg_realized_move_close  avg_straddle_edge_close
                0.056475                   0.034287                0.041931                0.007645                 0.046169                 0.011882

3. Quantiles of straddle_edge_open
 min_edge       p05       p25  median      p75     p95  max_edge
-0.145706 -0.051504 -0.016555 0.00405 0.027806 0.07172  0.230728

4. Spread diagnostics
 avg_before_call_spread_pct  avg_before_put_spread_pct  avg_after_call_spread_pct  avg_after_put_spread_pct
                   0.755907                   0.789367                   0.099952                  0.112501


In [3]:

import duckdb
import os

con = duckdb.connect()

PROCESSED_DIR = r"data\processed"

base_file = os.path.join(PROCESSED_DIR, "earnings_model_base.parquet")

clean_loose_file = os.path.join(PROCESSED_DIR, "earnings_model_clean_loose.parquet")
clean_strict_file = os.path.join(PROCESSED_DIR, "earnings_model_clean_strict.parquet")

# Keep the old name too, so your later notebook cells still work.
clean_file = os.path.join(PROCESSED_DIR, "earnings_model_clean.parquet")

for f in [clean_loose_file, clean_strict_file, clean_file]:
    if os.path.exists(f):
        os.remove(f)

con.execute(f"""
CREATE OR REPLACE TEMP TABLE base AS
SELECT *
FROM read_parquet('{base_file}')
""")

# ---------------------------------------------------------
# Loose filters:
# Good for getting a bigger sample.
# Does not require same-day volume.
# ---------------------------------------------------------
con.execute("""
CREATE OR REPLACE TEMP TABLE earnings_model_clean_loose AS
SELECT *
FROM base
WHERE
    implied_move_isolated IS NOT NULL
AND realized_move_open_to_prevclose IS NOT NULL

-- spread filters
AND before_call_spread_pct < 0.75
AND before_put_spread_pct < 0.75
AND after_call_spread_pct < 0.50
AND after_put_spread_pct < 0.50

-- open interest filters
AND COALESCE(before_call_oi, 0) >= 1
AND COALESCE(before_put_oi, 0) >= 1
AND COALESCE(after_call_oi, 0) >= 1
AND COALESCE(after_put_oi, 0) >= 1
""")

# ---------------------------------------------------------
# Strict filters:
# Similar to your original final clean dataset.
# ---------------------------------------------------------
con.execute("""
CREATE OR REPLACE TEMP TABLE earnings_model_clean_strict AS
SELECT *
FROM base
WHERE
    implied_move_isolated IS NOT NULL
AND realized_move_open_to_prevclose IS NOT NULL

-- spread filters
AND before_call_spread_pct < 0.5
AND before_put_spread_pct < 0.5
AND after_call_spread_pct < 0.3
AND after_put_spread_pct < 0.3

-- open interest filters
AND COALESCE(before_call_oi, 0) >= 10
AND COALESCE(before_put_oi, 0) >= 10
AND COALESCE(after_call_oi, 0) >= 10
AND COALESCE(after_put_oi, 0) >= 10

-- volume filters
AND COALESCE(before_call_vol, 0) >= 1
AND COALESCE(before_put_vol, 0) >= 1
AND COALESCE(after_call_vol, 0) >= 1
AND COALESCE(after_put_vol, 0) >= 1
""")

# Save both versions
con.execute(f"""
COPY earnings_model_clean_loose
TO '{clean_loose_file}'
(FORMAT PARQUET)
""")

con.execute(f"""
COPY earnings_model_clean_strict
TO '{clean_strict_file}'
(FORMAT PARQUET)
""")

# Save strict version using the old filename too, for compatibility.
con.execute(f"""
COPY earnings_model_clean_strict
TO '{clean_file}'
(FORMAT PARQUET)
""")

print(f"Saved loose file:  {clean_loose_file}")
print(f"Saved strict file: {clean_strict_file}")
print(f"Saved old-name strict file: {clean_file}")

# ---------------------------------------------------------
# Diagnostics
# ---------------------------------------------------------

def show(title, query, n=20):
    print("\n" + "="*100)
    print(title)
    print("="*100)
    df = con.execute(query).df()
    if len(df) > n:
        print(df.head(n).to_string(index=False))
        print(f"\n[showing first {n} rows out of {len(df)}]")
    else:
        print(df.to_string(index=False))

show(
"1. Row counts: base vs loose vs strict",
"""
SELECT 'base' AS dataset, COUNT(*) AS n_rows FROM base
UNION ALL
SELECT 'loose' AS dataset, COUNT(*) AS n_rows FROM earnings_model_clean_loose
UNION ALL
SELECT 'strict' AS dataset, COUNT(*) AS n_rows FROM earnings_model_clean_strict
"""
)

show(
"2. Edge distribution: loose",
"""
SELECT
AVG(straddle_edge_open) AS avg_edge,
APPROX_QUANTILE(straddle_edge_open,0.05) AS p05,
APPROX_QUANTILE(straddle_edge_open,0.25) AS p25,
APPROX_QUANTILE(straddle_edge_open,0.5) AS median,
APPROX_QUANTILE(straddle_edge_open,0.75) AS p75,
APPROX_QUANTILE(straddle_edge_open,0.95) AS p95
FROM earnings_model_clean_loose
"""
)

show(
"3. Edge distribution: strict",
"""
SELECT
AVG(straddle_edge_open) AS avg_edge,
APPROX_QUANTILE(straddle_edge_open,0.05) AS p05,
APPROX_QUANTILE(straddle_edge_open,0.25) AS p25,
APPROX_QUANTILE(straddle_edge_open,0.5) AS median,
APPROX_QUANTILE(straddle_edge_open,0.75) AS p75,
APPROX_QUANTILE(straddle_edge_open,0.95) AS p95
FROM earnings_model_clean_strict
"""
)

show(
"4. Year coverage: strict",
"""
SELECT
EXTRACT(YEAR FROM rdq) AS year,
COUNT(*) AS n_events
FROM earnings_model_clean_strict
GROUP BY 1
ORDER BY 1
"""
)

show(
"5. Sample strict rows",
"""
SELECT
tic,
rdq,
implied_move_simple,
implied_move_isolated,
realized_move_open_to_prevclose,
straddle_edge_open
FROM earnings_model_clean_strict
ORDER BY rdq, tic
LIMIT 30
"""
)


Saved loose file:  data\processed\earnings_model_clean_loose.parquet
Saved strict file: data\processed\earnings_model_clean_strict.parquet
Saved old-name strict file: data\processed\earnings_model_clean.parquet

1. Row counts: base vs loose vs strict
dataset  n_rows
   base     955
  loose     437
 strict     329

2. Edge distribution: loose
 avg_edge       p05       p25   median      p75      p95
 0.020211 -0.023245 -0.000677 0.015568 0.035664 0.072838

3. Edge distribution: strict
 avg_edge       p05     p25   median      p75      p95
 0.020279 -0.014087 0.00079 0.015875 0.036275 0.067319

4. Year coverage: strict
 year  n_events
 2018        37
 2019        47
 2020        37
 2021        50
 2022        55
 2023        51
 2024        52

5. Sample strict rows
 tic        rdq  implied_move_simple  implied_move_isolated  realized_move_open_to_prevclose  straddle_edge_open
  DE 2018-02-16             0.054253               0.006145                         0.013368            0.007224

In [4]:

import duckdb
import os

con = duckdb.connect()

PROCESSED_DIR = r"data\processed"

# Use strict by default.
# You can change this to earnings_model_clean_loose.parquet if strict has too few rows.
clean_file = os.path.join(PROCESSED_DIR, "earnings_model_clean_strict.parquet")

con.execute(f"""
CREATE OR REPLACE TEMP TABLE clean AS
SELECT *
FROM read_parquet('{clean_file}')
""")

def show(title, query, n=20):
    print("\n" + "="*100)
    print(title)
    print("="*100)
    df = con.execute(query).df()
    if len(df) > n:
        print(df.head(n).to_string(index=False))
        print(f"\n[showing first {n} rows out of {len(df)}]")
    else:
        print(df.to_string(index=False))

# ============================================================
# Final diagnostics
# ============================================================

show(
"1. Implied move distribution",
"""
SELECT
COUNT(*) AS n_events,
AVG(implied_move_isolated) AS avg_implied_move_isolated,
AVG(implied_move_simple) AS avg_implied_move_simple,
APPROX_QUANTILE(implied_move_isolated,0.05) AS p05,
APPROX_QUANTILE(implied_move_isolated,0.25) AS p25,
APPROX_QUANTILE(implied_move_isolated,0.5) AS median,
APPROX_QUANTILE(implied_move_isolated,0.75) AS p75,
APPROX_QUANTILE(implied_move_isolated,0.95) AS p95,
MAX(implied_move_isolated) AS max_implied_move
FROM clean
"""
)

show(
"2. Realized move distribution",
"""
SELECT
COUNT(*) AS n_events,
AVG(realized_move_open_to_prevclose) AS avg_realized_move_open,
APPROX_QUANTILE(realized_move_open_to_prevclose,0.05) AS p05,
APPROX_QUANTILE(realized_move_open_to_prevclose,0.25) AS p25,
APPROX_QUANTILE(realized_move_open_to_prevclose,0.5) AS median,
APPROX_QUANTILE(realized_move_open_to_prevclose,0.75) AS p75,
APPROX_QUANTILE(realized_move_open_to_prevclose,0.95) AS p95,
MAX(realized_move_open_to_prevclose) AS max_realized_move
FROM clean
"""
)

show(
"3. Edge distribution",
"""
SELECT
COUNT(*) AS n_events,
AVG(straddle_edge_open) AS avg_edge,
APPROX_QUANTILE(straddle_edge_open,0.05) AS p05,
APPROX_QUANTILE(straddle_edge_open,0.25) AS p25,
APPROX_QUANTILE(straddle_edge_open,0.5) AS median,
APPROX_QUANTILE(straddle_edge_open,0.75) AS p75,
APPROX_QUANTILE(straddle_edge_open,0.95) AS p95
FROM clean
"""
)

show(
"4. Events by year",
"""
SELECT
EXTRACT(YEAR FROM rdq) AS year,
COUNT(*) AS n_events
FROM clean
GROUP BY 1
ORDER BY 1
"""
)

show(
"5. Most common tickers",
"""
SELECT
tic,
COUNT(*) AS n_events
FROM clean
GROUP BY tic
ORDER BY n_events DESC
LIMIT 20
"""
)



1. Implied move distribution
 n_events  avg_implied_move_isolated  avg_implied_move_simple      p05      p25   median      p75      p95  max_implied_move
      329                   0.014656                 0.049417 0.004521 0.007346 0.010485 0.015697 0.037111          0.187117

2. Realized move distribution
 n_events  avg_realized_move_open      p05      p25   median      p75      p95  max_realized_move
      329                0.034935 0.002478 0.013655 0.029523 0.047827 0.082299           0.238031

3. Edge distribution
 n_events  avg_edge       p05     p25   median      p75      p95
      329  0.020279 -0.014087 0.00079 0.015875 0.036275 0.067319

4. Events by year
 year  n_events
 2018        37
 2019        47
 2020        37
 2021        50
 2022        55
 2023        51
 2024        52

5. Most common tickers
 tic  n_events
 SLB        26
 CVX        26
 XOM        24
 WFC        16
ABBV        16
   C        15
  CL        15
  DE        15
 JPM        14
 UNH         8
  WY 